# Pro Cycling Intelligence Agent — Playground

Test every phase of the agent without Telegram. Each cell is independent so you can inspect exactly what happens at each step.

```
Question → [Planner] → ResearchPlan → [Executor] → Raw Findings → [Synthesizer] → CyclingAnswer
```

In [ ]:
# Run once to install all dependencies
%pip install anthropic procyclingstats tavily-python httpx beautifulsoup4 pydantic python-dotenv rich

---
## Step 0 — Setup

Keys are loaded from the nearest `.env` file (either `cycling_agent/.env` or the parent `AI_Agent/.env`).
You don't need `TELEGRAM_BOT_TOKEN` for the notebook — only `ANTHROPIC_API_KEY` and `TAVILY_API_KEY`.

In [ ]:
import os
import sys
from dotenv import load_dotenv, find_dotenv

# Add cycling_agent/ to path so imports work regardless of where Jupyter was launched
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

# find_dotenv() walks up the directory tree to find the nearest .env
load_dotenv(find_dotenv())

anthropic_ok = bool(os.environ.get("ANTHROPIC_API_KEY"))
tavily_ok    = bool(os.environ.get("TAVILY_API_KEY"))

print(f"ANTHROPIC_API_KEY : {'OK' if anthropic_ok else 'MISSING — check your .env'}")
print(f"TAVILY_API_KEY    : {'OK' if tavily_ok    else 'MISSING — check your .env'}")

---
## Step 1 — The Data Models

Two schemas drive the whole agent:
- `ResearchPlan` — what the Planner outputs (which tools to call and with what queries)
- `CyclingAnswer` — what the Synthesizer outputs (the final structured answer)

Both are passed to Claude as tool definitions. Claude fills them in like a form — no free-text parsing needed.

In [ ]:
from models.plan import ResearchPlan, CyclingToolType
from models.answer import CyclingAnswer
import json

print("=== CyclingToolType — all available tools ===")
for t in CyclingToolType:
    print(f"  {t.value}")

print()
print("=== ResearchPlan schema (sent to Claude as a tool) ===")
print(json.dumps(ResearchPlan.model_json_schema(), indent=2))

In [ ]:
print("=== CyclingAnswer schema (sent to Claude as a tool) ===")
print(json.dumps(CyclingAnswer.model_json_schema(), indent=2))

---
## Step 2 — The PCS Tools (no AI)

These are plain Python wrappers around procyclingstats.com. No Claude, no randomness.
Test them directly here to see the raw structured data before the agent processes it.

In [ ]:
from tools.cycling_pcs import get_individual_ranking, get_team_ranking

print("=== Top 5 UCI WorldTour individual ranking ===")
ranking = get_individual_ranking(top_n=5)
for r in ranking:
    print(r)

In [ ]:
from tools.cycling_pcs import get_rider_profile

# Change the slug to any rider — format: firstname-lastname
rider_slug = "tadej-pogacar"
print(f"=== Rider profile: {rider_slug} ===")
profile = get_rider_profile(rider_slug)
print(json.dumps(profile, indent=2, default=str))

In [ ]:
from tools.cycling_pcs import get_race_overview

# Format: get_race_overview(race_slug, year)
race = get_race_overview("tour-de-france", 2024)
print("=== Tour de France 2024 overview ===")
print(json.dumps(race, indent=2, default=str))

In [ ]:
from tools.cycling_pcs import get_stage_results

# Format: get_stage_results(race_slug, year, stage_number)
stage = get_stage_results("tour-de-france", 2024, 1)
print("=== Tour de France 2024 — Stage 1 results ===")
print(json.dumps(stage, indent=2, default=str))

In [ ]:
from tools.cycling_pcs import get_rider_results

# Format: get_rider_results(rider_slug, year)
results = get_rider_results("tadej-pogacar", 2024)
print("=== Pogacar 2024 results (first 5) ===")
for r in results[:5]:
    print(r)

### Web search fallback
Used when PCS doesn't have the data — live race updates, breaking news, etc.

In [ ]:
from tools.web_search import search

results = search("Tour de France 2025 stage results latest", max_results=3)
for r in results:
    print(f"{r.title}\n{r.url}\n{r.content[:200]}\n")

---
## Step 3 — The Planner

Claude reads your question and decides which tools to call and in what order.
It outputs a structured `ResearchPlan` by being forced to call a tool — no free text.

**Change the question here:**

In [ ]:
QUESTION = "Who is currently leading the WorldTour standings?"  # <- change this

In [ ]:
from agent.planner import create_plan

print(f"Question: {QUESTION}\n")
plan = create_plan(QUESTION)

print(f"Claude generated {len(plan.steps)} steps:\n")
for step in plan.steps:
    print(f"  Step {step.step_id} [{step.tool.value}]")
    print(f"    Goal : {step.description}")
    print(f"    Query: {step.query}")
    print()

**Notice:** For a cycling question, Claude picks PCS tools over web search when possible — structured data is more reliable than scraped text. Run the cell twice and the steps may vary slightly (Claude is non-deterministic).

---
## Step 4 — The Executor

A plain `for` loop — no AI. Reads each step, calls the right tool, collects raw data.
The output is a dict: `{ topic description → raw JSON or text }`

In [ ]:
from agent.executor import execute_plan

print(f"Executing {len(plan.steps)} steps...\n")
raw_findings = execute_plan(plan)

print("Topics collected:")
for topic in raw_findings:
    print(f"  - {topic}")

In [ ]:
# Inspect raw data for any topic — change [0] to explore others
topic = list(raw_findings.keys())[0]
print(f"=== Raw findings for: '{topic}' ===\n")
print(raw_findings[topic][:2000])

---
## Step 5 — The Synthesizer

Claude reads all raw findings and produces a typed `CyclingAnswer`.
Same tool-use trick as the Planner — Claude fills in the schema fields, Pydantic validates them.

In [ ]:
from agent.synthesizer import synthesize

print("Synthesizing answer...\n")
answer = synthesize(QUESTION, raw_findings)

print("Raw JSON output from Claude:")
print(answer.model_dump_json(indent=2))

---
## Step 6 — Rendered Report

The same output rendered as a readable report — this is what the Telegram bot sends (minus the HTML tags).

In [ ]:
from IPython.display import display, Markdown

def report(a) -> None:
    confidence_icon = {"high": "🟢", "medium": "🟡", "low": "🔴"}.get(a.confidence, "⚪")
    data_points = "\n".join(f"- {p}" for p in a.data_points) or "_None_"
    follow_ups  = "\n".join(f"{i}. {s}" for i, s in enumerate(a.follow_up_suggestions, 1)) or "_None_"
    source      = f"_{a.source_note}_" if a.source_note else ""

    md = f"""
---
## 🚴 {a.question}

{a.answer}

### 📊 Key Facts
{data_points}

### 💡 You might also ask
{follow_ups}

**Confidence:** {confidence_icon} {a.confidence.upper()}

{source}

---
"""
    display(Markdown(md))


report(answer)

---
## Step 7 — Full Pipeline

Everything in one function — identical to what `main.py` and the Telegram bot run.

In [ ]:
def ask(question: str):
    print(f"[1/3] Planning for: '{question}'...")
    plan = create_plan(question)
    print(f"      {len(plan.steps)} steps — {', '.join(s.tool.value for s in plan.steps)}")

    print("[2/3] Executing steps...")
    findings = execute_plan(plan)
    print(f"      {len(findings)} topics collected")

    print("[3/3] Synthesizing answer...")
    answer = synthesize(question, findings)
    print(f"      Confidence: {answer.confidence}\n")

    report(answer)
    return answer


# Try it
result = ask("What are Tadej Pogacar's biggest wins in 2024?")

---
## Experiment Zone

Ideas to explore:

1. **Ask about a live race** — Claude will use `search` instead of PCS tools
2. **Ask about an obscure rider** — confidence drops to `medium` or `low`
3. **Ask a stage question** — watch the planner pick `pcs_stage` with the right slug/year/stage format
4. **Inspect raw PCS data** before and after synthesis — see how much Claude extracts vs ignores
5. **Edit the synthesizer system prompt** in `agent/synthesizer.py` — change the tone (e.g. more technical, shorter answers)

In [ ]:
# Free sandbox — ask anything
ask("Who won stage 5 of the Giro d'Italia 2024?")

In [ ]:
ask("Show me the top 10 UCI WorldTour team rankings")

In [ ]:
ask("What is Remco Evenepoel's riding style and what races suit him best?")